In [0]:
%pip install -U langchain langchain-community databricks-langchain langchain_chroma pypdf
 
dbutils.library.restartPython()

In [0]:
from databricks_langchain import ChatDatabricks, DatabricksEmbeddings
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [0]:

# ==========================================
# 1. DATA PREPROCESSING
# ==========================================

loader = DirectoryLoader(
    "/Volumes/dev/bronze/raw/resumes/",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20
)

chunks = text_splitter.split_documents(documents)

print(f"Loaded pages: {len(documents)}")
print(f"Created chunks: {len(chunks)}")


# ==========================================
# 2. EMBEDDINGS
# ==========================================

embeddings = DatabricksEmbeddings(
    endpoint="databricks-bge-large-en"
)


# ==========================================
# 3. VECTOR STORE
# ==========================================

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)


# ==========================================
# 4. RETRIEVER
# ==========================================

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)


# ==========================================
# 5. LLM
# ==========================================

llm = ChatDatabricks(
    model="databricks-gpt-oss-20b"
)


# ==========================================
# 6. PROMPT
# ==========================================

prompt_template = PromptTemplate(
    input_variables=["input", "context"],
    template="""You are an AI Resume Screening Assistant designed to help HR professionals evaluate candidates.

Your task is to answer questions based ONLY on the provided resume context and its metadata.

Guidelines:

1. Answer accurately using the resume content and metadata.
2. Never invent candidate information.
3. Candidate names may be available in the document metadata under:
   candidate_name
4. If multiple resumes are present, identify and list each unique candidate separately.
5. When asked for candidate names, return all candidate names available in the retrieved context.
6. When asked about a specific candidate, use the candidate_name metadata to identify the correct resume.
7. For candidate suitability questions, clearly distinguish between:
   - Matching skills and experience
   - Missing or unclear requirements
8. If information is not available, say:
   "This information is not mentioned in the resume."
9. Do not make decisions based on sensitive personal characteristics.
10. Keep responses professional and concise.
11. Use bullet points when listing multiple candidates.

Resume Context:
{context}

HR Question:
{input}

Answer:"""
)

# ==========================================
# 7. QA CHAIN
# ==========================================

qa_chain = create_stuff_documents_chain(
    llm,
    prompt_template
)


# ==========================================
# 8. RAG CHAIN
# ==========================================

rag_chain = create_retrieval_chain(
    retriever,
    qa_chain
)


# ==========================================
# 9. CHAT
# ==========================================

print("\nChat with your own Data")

question = input("What is your query? ")

if question:
    response = rag_chain.invoke({
        "input": question
    })

    print("\nAnswer:")
    answer = response["answer"]

if isinstance(answer, list):
    text_parts = [
        item["text"]
        for item in answer
        if isinstance(item, dict) and item.get("type") == "text"
    ]
    answer = "\n".join(text_parts)

print(answer)

In [0]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

# Resume folder path
RESUME_PATH = "/Volumes/dev/bronze/raw/resumes/"

# Find all PDF files
pdf_files = list(Path(RESUME_PATH).glob("*.pdf"))

print(f"Found {len(pdf_files)} resume(s)")

documents = []

for pdf_file in pdf_files:

    print(f"Loading: {pdf_file.name}")

    loader = PyPDFLoader(str(pdf_file))
    docs = loader.load()

    # Candidate identifier from filename
    candidate_name = pdf_file.stem

    # Add metadata to every page
    for doc in docs:
        doc.metadata["candidate_name"] = candidate_name
        doc.metadata["source_file"] = pdf_file.name

    documents.extend(docs)

print(f"\nTotal pages loaded: {len(documents)}")

for pdf_file in pdf_files:
    print(f" - {pdf_file.stem}")

In [0]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

In [0]:
# ============================================================
# 4. EMBEDDINGS
# ============================================================

embeddings = DatabricksEmbeddings(
    endpoint="databricks-bge-large-en"
)

In [0]:
# ============================================================
# 5. VECTOR STORE
# ============================================================

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="resume_hr_agent"
)

print("Vector store created successfully")

In [0]:
# ============================================================
# 7. LLM
# ============================================================

llm = ChatDatabricks(
    model="databricks-gemma-3-12b"
)

In [0]:
# ============================================================
# 8. HR RESUME PROMPT
# ============================================================

prompt_template = PromptTemplate(
    input_variables=["input", "context"],
    template="""
You are an AI Resume Screening Assistant designed for HR and recruitment teams.

You are given information retrieved from multiple candidate resumes.

Your job is to answer the HR user's question using ONLY the information available in the provided context.

IMPORTANT RULES:

1. The context may contain information from MULTIPLE resumes.

2. Each resume chunk contains metadata such as:
   - candidate_name
   - source_file

3. Treat candidate_name as the candidate identifier.

4. When the user asks for candidate names:
   - Identify ALL unique candidates available in the context.
   - Do not return only the first candidate.
   - Return each candidate only once.

5. When the user asks about a specific candidate:
   - Use the candidate_name to identify that candidate's information.

6. When comparing candidates:
   - Keep information separated by candidate.
   - Never mix experience, skills, projects, or certifications between candidates.

7. Never invent information.

8. If information is not available in the retrieved context, say:
   "This information is not available in the retrieved resumes."

9. Do NOT reveal your reasoning or internal thought process.

10. Return ONLY the final answer.

11. For multiple candidates, use a numbered list or table.

12. For a yes/no question:
   - Start with Yes or No.
   - Then give a short explanation.

13. For candidate suitability:
   - Mention matching skills/experience.
   - Mention missing or unclear requirements.
   - Do not make decisions based on sensitive personal characteristics.

14. Keep the answer professional and concise.

-----------------------------------------
RESUME CONTEXT
-----------------------------------------

{context}

-----------------------------------------
HR QUESTION
-----------------------------------------

{input}

-----------------------------------------
FINAL ANSWER
-----------------------------------------
"""
)

In [0]:
# ============================================================
# 9. DOCUMENT QA CHAIN
# ============================================================

qa_chain = create_stuff_documents_chain(
    llm,
    prompt_template
)

In [0]:
# ============================================================
# 10. RETRIEVAL RAG CHAIN
# ============================================================

rag_chain = create_retrieval_chain(
    retriever,
    qa_chain
)

In [0]:
# ============================================================
# 11. CLEAN LLM RESPONSE
# ============================================================

def extract_answer(answer):

    # Normal string response
    if isinstance(answer, str):
        return answer.strip()

    # GPT-OSS structured response
    if isinstance(answer, list):

        text_parts = []

        for item in answer:

            if isinstance(item, dict):

                if item.get("type") == "text":
                    text = item.get("text", "")

                    if text:
                        text_parts.append(text)

        return "\n".join(text_parts).strip()

    return str(answer).strip()

In [0]:
# ============================================================
# 12. CHAT WITH RESUMES
# ============================================================

print("\n========================================")
print("      RESUME HR ASSISTANT")
print("========================================")
print(f"Candidates loaded: {len(pdf_files)}")

for pdf_file in pdf_files:
    print(f" - {pdf_file.stem}")

print("\nType 'exit' to stop.\n")


while True:

    question = input("HR Query: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    if not question.strip():
        continue

    response = rag_chain.invoke({
        "input": question
    })

    answer = extract_answer(response["answer"])

    print("\nAnswer:")
    print(answer)

    print("\n" + "-" * 70 + "\n")

In [0]:
# ============================================================
# RESUME HR RAG AGENT
# ============================================================


# ============================================================
# 1. CONFIGURATION
# ============================================================

RESUME_PATH = "/Volumes/dev/bronze/raw/resumes/"

EMBEDDING_ENDPOINT = "databricks-bge-large-en"

# Use your Gemma endpoint here
LLM_MODEL = "databricks-gemma-3-12b"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 10


# ============================================================
# 2. LOAD ALL PDF RESUMES
# ============================================================

print("=" * 60)
print("RESUME HR RAG AGENT")
print("=" * 60)

pdf_files = list(Path(RESUME_PATH).glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in: {RESUME_PATH}"
    )

print(f"\nFound {len(pdf_files)} resume(s):")

for pdf in pdf_files:
    print(f"  - {pdf.name}")


documents = []

candidate_names = []


# ============================================================
# 3. LOAD PDFs
# ============================================================

for pdf_file in pdf_files:

    print(f"\nLoading: {pdf_file.name}")

    loader = PyPDFLoader(str(pdf_file))
    docs = loader.load()

    # --------------------------------------------------------
    # Get text from first page
    # --------------------------------------------------------

    first_page_text = ""

    if docs:
        first_page_text = docs[0].page_content[:2000]


    # --------------------------------------------------------
    # Candidate name
    #
    # For now use filename.
    # The first-page text is also stored in metadata so the
    # LLM can identify the actual name from the resume.
    # --------------------------------------------------------

    candidate_name = pdf_file.stem

    for doc in docs:

        doc.metadata["candidate_name"] = candidate_name
        doc.metadata["source_file"] = pdf_file.name

    documents.extend(docs)

    candidate_names.append(candidate_name)


print(f"\nTotal pages loaded: {len(documents)}")


# ============================================================
# 4. SHOW CANDIDATES
# ============================================================

candidate_names = sorted(set(candidate_names))

print("\nCandidates detected:")

for name in candidate_names:
    print(f"  - {name}")


# ============================================================
# 5. CHUNKING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = text_splitter.split_documents(documents)

print(f"\nTotal chunks created: {len(chunks)}")


# ============================================================
# 6. EMBEDDINGS
# ============================================================

print("\nCreating embeddings...")

embeddings = DatabricksEmbeddings(
    endpoint=EMBEDDING_ENDPOINT
)


# ============================================================
# 7. VECTOR STORE
# ============================================================

print("Creating Chroma vector store...")

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="resume_hr_agent"
)


# ============================================================
# 8. RETRIEVER
# ============================================================

retriever = vector_store.as_retriever(
    search_kwargs={
        "k": TOP_K
    }
)


# ============================================================
# 9. LLM
# ============================================================

print("Loading LLM...")

llm = ChatDatabricks(
    model=LLM_MODEL
)


# ============================================================
# 10. HR PROMPT
# ============================================================

prompt_template = PromptTemplate(
    input_variables=["input", "context"],
    template="""
You are an AI Resume Screening Assistant for HR professionals.

You are answering questions about multiple candidate resumes.

Use ONLY the information provided in the resume context.

IMPORTANT RULES:

1. The context can contain information from multiple candidates.

2. Never mix information between candidates.

3. Every resume chunk has metadata:
   candidate_name
   source_file

4. If asked about a candidate, identify the correct candidate.

5. If asked to compare candidates, keep each candidate's information separate.

6. Never invent information.

7. If information is unavailable, say:
   "This information is not available in the resumes."

8. Never reveal internal reasoning.

9. Return ONLY the final answer.

10. Keep answers professional and concise.

11. For multiple candidates, use a numbered list or table.

12. For skills/experience questions, mention the candidate name.

13. For candidate suitability, mention:
   - Matching skills
   - Relevant experience
   - Missing or unclear requirements

14. Do not use sensitive personal characteristics for candidate evaluation.

Resume Context:
{context}

HR Question:
{input}

Final Answer:
"""
)


# ============================================================
# 11. CHAINS
# ============================================================

qa_chain = create_stuff_documents_chain(
    llm,
    prompt_template
)

rag_chain = create_retrieval_chain(
    retriever,
    qa_chain
)


# ============================================================
# 12. CLEAN MODEL RESPONSE
# ============================================================

def clean_answer(answer):

    if isinstance(answer, str):
        return answer.strip()

    if isinstance(answer, list):

        text_parts = []

        for item in answer:

            if isinstance(item, dict):

                if item.get("type") == "text":

                    text = item.get("text", "")

                    if text:
                        text_parts.append(text)

        return "\n".join(text_parts).strip()

    return str(answer).strip()


# ============================================================
# 13. DETECT "ALL CANDIDATES" QUESTIONS
# ============================================================

def is_candidate_list_question(question):

    q = question.lower().strip()

    # Normalize punctuation
    q = re.sub(r"[^a-z0-9\s]", " ", q)

    # Common candidate-list questions
    patterns = [
        "candidate names",
        "candidates names",
        "candidate name",
        "all candidates",
        "all candidate",
        "list candidates",
        "list candidate",
        "list of candidates",
        "list all candidates",
        "show candidates",
        "show all candidates",
        "give candidates",
        "give me candidates",
        "give me all candidates",
        "who are the candidates",
        "how many candidates",
        "all resumes",
        "all resume",
        "candidate resumes",
        "candidates resumes",
        "all candidate resumes"
    ]

    return any(pattern in q for pattern in patterns)


# ============================================================
# 14. CHAT
# ============================================================

print("\n" + "=" * 60)
print("HR RESUME ASSISTANT READY")
print("=" * 60)

print("\nExample questions:")
print("  - Give me all candidates")
print("  - All candidates resume?")
print("  - Give me the candidate names")
print("  - Who has Databricks experience?")
print("  - Who has Python and PySpark experience?")
print("  - What certifications does Prabhakar have?")
print("  - Compare the candidates")
print("  - Which candidates have GenAI experience?")

print("\nType 'exit' to stop.\n")


while True:

    question = input("Query: ")

    if question.lower().strip() == "exit":
        print("\nGoodbye!")
        break

    if not question.strip():
        continue


    # ========================================================
    # ALL-CANDIDATE QUESTION
    # ========================================================

    if is_candidate_list_question(question):

        print("\nAnswer:")

        for i, name in enumerate(candidate_names, start=1):
            print(f"{i}. {name}")

        print("\n" + "-" * 60 + "\n")

        continue


    # ========================================================
    # NORMAL RAG QUESTION
    # ========================================================

    response = rag_chain.invoke({
        "input": question
    })

    answer = clean_answer(
        response["answer"]
    )

    print("\nAnswer:")
    print(answer)

    print("\n" + "-" * 60 + "\n")